# Fig S3: SER-fallback thresholds (data processing)

GTEx eQTL simulation (n = 800, Fig 3A) with ADSP, UKB and 1000 Genomes LD. SuSiE-rRSS falls back to the SER fit when `R_reliability_flag` is set: the artifact score `Q_art` exceeds `artifact_threshold` (default 0.1) or a credible set's log-BF attenuation `D_CS_max` reaches `sensitivity_threshold` (default 30). Both only set the flag after fitting, so each setting is re-applied to the stored fits (`Gtex_fine/summary`, `susie_bias_summary` from `10`); one threshold is varied at a time with the other at its default. `artifact = 1` / `sensitivity = Off` turns that check off.

**Output:** `figure_data/FigED1AB.SER_Fallback_Threshold.rds`: power, FDR (bootstrap 95% CI) and fraction of regions using the SER fallback, by threshold × reference × NCP

In [1]:
source("../../R/utils.R")

data_name <- "FigED1AB.SER_Fallback_Threshold"
n_GWAS    <- 800                     # GTEx eQTL n; NCP = n * h2g
refs      <- c("ADSP", "UKB", "kg")   # external references
keys      <- c("input_file", "ncausal", "reference")

res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine/summary/result_susie_bias_summary_summary.rds") %>%
  mutate(
    reference = str_extract(method, paste0("^(", paste(c("insample", refs), collapse = "|"), ")")),
    variant   = str_remove(method, paste0("^(", paste(c("insample", refs), collapse = "|"), ")")),
    variant   = ifelse(variant == "", "baseline", str_remove(variant, "^_"))   # baseline / SER / mix / mix_SER
  ) %>%
  filter(reference %in% refs)

ser_variant <- "SER"      # <-- plain {ref}_SER, as in Fig S3E. "mix_SER" for the EB-mix SER fallback.

mix <- filter(res, variant == "mix")
ser <- filter(res, variant == ser_variant)

In [2]:
## ---- fallback rule, re-applied at any thresholds -----------------------------
## susieR: R_reliability_flag = (artifact evaluable & Q_art > artifact_threshold) |
##                              (max CS log-BF attenuation >= sensitivity_threshold)
## Both thresholds only set the flag after fitting, so the stored fits are re-used.
fallback_flag <- function(d, artifact, sensitivity)
  (d$artifact_evaluable %in% TRUE & d$Q_art > artifact) |
  (is.finite(d$D_CS_max) & d$D_CS_max >= sensitivity)

## check: the defaults (0.1, 30) reproduce the stored flag
mix %>%
  mutate(recomputed = fallback_flag(mix, 0.1, 30), stored = R_reliability_flag %in% TRUE) %>%
  count(reference, recomputed, stored)

reference,recomputed,stored,n
<chr>,<lgl>,<lgl>,<int>
ADSP,FALSE,FALSE,7129
ADSP,TRUE,TRUE,71
UKB,FALSE,FALSE,6925
UKB,TRUE,TRUE,275
kg,FALSE,FALSE,6807
kg,TRUE,TRUE,393


In [3]:
## ---- threshold grid: vary one, keep the other at its default ------------------
grid <- bind_rows(
  tibble(sweep = "artifact",    artifact = c(0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1),   sensitivity = 30),
  tibble(sweep = "sensitivity", artifact = 0.1, sensitivity = c(log(3), log(20), log(150), 10, 20, 30, 50, 100, Inf))
) %>%
  mutate(setting = row_number(),
         is_default = artifact == 0.1 & sensitivity == 30)

## per region x reference: SER row if flagged (and present), EB-mix row otherwise
pick_rows <- function(artifact, sensitivity) {
  dec <- mix %>% transmute(across(all_of(keys)), fallback = fallback_flag(mix, artifact, sensitivity))
  use_ser <- dec %>% filter(fallback) %>% semi_join(ser, by = keys)
  bind_rows(
    ser %>% semi_join(use_ser, by = keys),
    mix %>% anti_join(use_ser, by = keys)
  ) %>%
    mutate(fallback = variant == ser_variant)
}

reps <- grid %>%
  mutate(rows = map2(artifact, sensitivity, pick_rows)) %>%
  unnest(rows) %>%
  mutate(h2g = as.numeric(str_extract(input_file, "(?<=h2g_)[0-9.]+")))

## ---- power / FDR with bootstrap CI (boot_cell_mix() in R/utils.R), + fallback rate
set.seed(1)
summ <- reps %>%
  group_by(sweep, setting, artifact, sensitivity, is_default, reference, h2g) %>%
  group_modify(~ bind_cols(boot_cell_mix(.x, 1000), tibble(fallback = mean(.x$fallback)))) %>%
  ungroup()

In [4]:
## ---- plot-ready ------------------------------------------------------------------
ref_lab <- c(ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes")
fmt_thr <- function(x) case_when(is.infinite(x) ~ "Off",
                                 abs(x - log(3))   < 1e-9 ~ "log 3",
                                 abs(x - log(20))  < 1e-9 ~ "log 20",
                                 abs(x - log(150)) < 1e-9 ~ "log 150",
                                 TRUE ~ as.character(x))

plot_df <- summ %>%
  mutate(
    threshold     = if_else(sweep == "artifact", artifact, sensitivity),
    threshold_lab = if_else(sweep == "artifact" & artifact == 1, "Off", fmt_thr(threshold)),
    sweep         = factor(sweep, levels = c("artifact", "sensitivity"),
                           labels = c("Artifact threshold (Q_art)", "Sensitivity threshold (CS log-BF attenuation)")),
    reference     = factor(ref_lab[reference], levels = unname(ref_lab)),
    ncp_num       = n_GWAS * h2g,
    ncp           = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 0)), ncp_num)
  ) %>%
  transmute(sweep, threshold, threshold_lab, is_default, reference, ncp,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est = FDR, FDR_lo = FDR_lo, FDR_hi = FDR_hi,
            `SER fallback_est` = fallback, `SER fallback_lo` = NA_real_, `SER fallback_hi` = NA_real_) %>%
  pivot_longer(c(starts_with("Power_"), starts_with("FDR_"), starts_with("SER fallback_")),
               names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(metric = factor(metric, levels = c("Power", "FDR", "SER fallback")))

save_data(plot_df, data_name)